In [4]:

import torch
import torch.nn as nn


class CNNTransformer(nn.Module):

    def __init__(
        self,
        num_classes=2,
        embed_dim=64,
        num_heads=4,
        num_layers=2,
        dropout=0.1
    ):
        super().__init__()

        # 1. CNN Feature Extractor
        self.cnn = nn.Sequential(

            nn.Conv3d(
                1, 32,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm3d(32),
            nn.GELU(),

            nn.Conv3d(
                32, embed_dim,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm3d(embed_dim),
            nn.GELU()
        )

        # 2. CLS Token
        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, embed_dim)
        )

        # 3. Positional Embedding
        self.pos_embedding = nn.Parameter(
            torch.zeros(1, 10, embed_dim)
        )

        # 4. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=128,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # 5. Classification Head
        self.classifier = nn.Sequential(

            nn.LayerNorm(embed_dim),

            nn.Linear(embed_dim, 32),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(32, num_classes)
        )

    def forward(self, x):

        # CNN feature extraction
        x = self.cnn(x)

        # Aggregate spectral dimension
        x = x.mean(dim=2)

        # Convert spatial features into tokens
        x = x.flatten(2).transpose(1, 2)

        # Add CLS token
        batch_size = x.size(0)

        cls_tokens = self.cls_token.expand(
            batch_size, -1, -1
        )

        x = torch.cat(
            (cls_tokens, x),
            dim=1
        )

        # Add positional embeddings
        x = x + self.pos_embedding

        # Transformer feature learning
        x = self.transformer(x)

        # CLS representation
        x = x[:, 0]

        # Classification
        output = self.classifier(x)

        return output


In [5]:

# from cnn_transformer import CNNTransformer

# Initialize model
model = CNNTransformer(num_classes=2)

# Demo hyperspectral input
# Batch, Channel, Spectral, Height, Width
x = torch.randn(8, 1, 16, 3, 3)

# Forward pass
output = model(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

print("Model architecture:")
print(model)

assert output.shape == (8, 2)

print("CNN + Transformer architecture test passed!")


Input shape: torch.Size([8, 1, 16, 3, 3])
Output shape: torch.Size([8, 2])
Model architecture:
CNNTransformer(
  (cnn): Sequential(
    (0): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (4): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): GELU(approximate='none')
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
      

# save model to srtifatcst

In [6]:
from pathlib import Path
import torch

# ARTIFACT_DIR = Path("backend/artifacts")
# ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

model_path = r"E:\internhips\infotact solutions\TerraSpectra Hyperspectral Crop Disease Forecasting\backend\artifacts\cnn_transformer_model.pth"

torch.save(
    model.state_dict(),
    model_path
)

print(f"Model saved: {model_path}")

Model saved: E:\internhips\infotact solutions\TerraSpectra Hyperspectral Crop Disease Forecasting\backend\artifacts\cnn_transformer_model.pth


# load the model

In [ ]:
model = CNNTransformer(num_classes=2)

model.load_state_dict(
    torch.load(
        r"E:\internhips\infotact solutions\TerraSpectra Hyperspectral Crop Disease Forecasting\backend\artifacts\cnn_transformer_model.pth",
        map_location="cpu",
        weights_only=True,
    )
)

model.eval()

CNNTransformer(
  (cnn): Sequential(
    (0): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (4): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): GELU(approximate='none')
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,

In [8]:
import numpy as np
import os

# Dataset path
dataset_path = r"E:\internhips\infotact solutions\TerraSpectra Hyperspectral Crop Disease Forecasting\backend\data\processed\saved_datasets\terraspectra_spatial_dataset.npz"

# Check if dataset exists
if not os.path.exists(dataset_path):
    raise FileNotFoundError(
        f"Dataset not found: {dataset_path}"
    )

# Load dataset
data = np.load(dataset_path)

# Extract training and testing data
X_train = data["X_train"]
X_test = data["X_test"]

y_train = data["y_train"]
y_test = data["y_test"]

# Extract spatial masks and indices
train_indices = data["train_indices"]
test_indices = data["test_indices"]

train_mask = data["train_mask"]
test_mask = data["test_mask"]
buffer_mask = data["buffer_mask"]

# Display dataset information
print("TerraSpectra Spatial Dataset Loaded Successfully!")

print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)

print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

print("\nTraining classes:")
print(np.unique(y_train, return_counts=True))

print("\nTesting classes:")
print(np.unique(y_test, return_counts=True))

TerraSpectra Spatial Dataset Loaded Successfully!

Training data: (864, 16)
Testing data: (72, 16)
Training labels: (864,)
Testing labels: (72,)

Training classes:
(array([0, 1], dtype=uint8), array([761, 103]))

Testing classes:
(array([0, 1], dtype=uint8), array([33, 39]))


In [9]:
import torch
with torch.no_grad():

    outputs = model(X_test)

    probabilities = torch.softmax(
        outputs, dim=1
    )

    predictions = torch.argmax(
        probabilities, dim=1
    )

print("Predictions:", predictions.shape)
print("Probabilities:", probabilities.shape)

TypeError: conv3d() received an invalid combination of arguments - got (numpy.ndarray, Parameter, Parameter, tuple, tuple, tuple, int), but expected one of:
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, tuple of ints padding = 0, tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!numpy.ndarray!, !Parameter!, !Parameter!, !tuple of (int, int, int)!, !tuple of (int, int, int)!, !tuple of (int, int, int)!, !int!)
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, str padding = "valid", tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!numpy.ndarray!, !Parameter!, !Parameter!, !tuple of (int, int, int)!, !tuple of (int, int, int)!, !tuple of (int, int, int)!, !int!)


# savre the pred

In [ ]:
import numpy as np

np.save(
    "backend/artifacts/cnn_transformer_predictions.npy",
    predictions.cpu().numpy()
)

np.save(
    "backend/artifacts/cnn_transformer_probabilities.npy",
    probabilities.cpu().numpy()
)